# Stored-weights KL check: all three models (Colab A100 80 GB)

Runs `weight_observer.packed_check` (commit `31b4a2e9a09b`) for the check in `docs/CHECK_packed_weights_kl.md`
(amendment 1: all three models on this GPU). For each model and each of the six GPTQ arms the product ships, it
encodes with the product encoder and measures the KL, then stores the weights as a `.tqpw` file, decodes them back and
measures again.

Inputs come from the repository at that commit (plans and registered results, sha-checked) plus the registered models,
text and windows (hash-checked). Outputs: `MyDrive/tqp-weights/packed/<model>`, one arm at a time. If Colab
disconnects, reconnect to an A100 80 GB and Run all again: finished arms are skipped.

**Runtime: A100 80 GB (High-RAM).** Then Runtime > Run all. About 10 min of setup, then roughly 2.5 hours of GPU work
(Llama-3.1-8B first, then Qwen2.5-3B, then Gemma-2-2B).

In [ ]:
# 1. GPU: the A100 80 GB, checked before anything is downloaded.
import os, subprocess, sys
TEST = os.environ.get("NB_TEST") == "1"
if not TEST:
    import torch
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout)
    gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"{gib:.1f} GiB")
    assert gib > 70, "not the 80 GB A100: Runtime > Disconnect and delete runtime, then reconnect"

In [ ]:
# 2. Paths, and Google Drive (outputs only).
ROOT = os.environ.get("NB_ROOT", "/content")
OUT = os.environ.get("NB_OUT", "/content/drive/MyDrive/tqp-weights/packed")
if not TEST:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
print("outputs", OUT)

In [ ]:
# 3. The pinned packages (as registered: torch 2.8.0 with torchvision 0.23.0, transformers 4.56.1).
if not TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torch==2.8.0", "torchvision==0.23.0",
                    "transformers==4.56.1", "accelerate==1.14.0", "datasets==4.3.0", "safetensors", "numpy",
                    "huggingface_hub"], check=True)
PY = os.environ.get("NB_PY", sys.executable)
print(subprocess.run([PY, "-c", "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# 4. The code: a fresh full clone in a commit-named folder, the product package and the harness importable.
C = "31b4a2e9a09bd40fd210bbabadbf73d8a5d4cfc2"
CODE = f"{ROOT}/tqp_{C[:12]}"
if not os.path.exists(f"{CODE}/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/ahb-sjsu/turboquant-pro", CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", "--detach", C], check=True)
assert subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == C
ENV = dict(os.environ, PYTHONPATH=f"{CODE}:{CODE}/benchmarks", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_XET="1",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")


def step(args, label):
    print(">>", label, flush=True)
    p = subprocess.Popen([PY, *args], cwd=f"{CODE}/benchmarks", env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"step failed ({p.returncode}): {label}" + chr(10) + "".join(lines[-30:]))


step(["-c", "import turboquant_pro.weight_codec, turboquant_pro.packed_weights, weight_observer.packed_check; print('imports ok')"],
     "imports")
print("code", C[:12])

In [ ]:
# 5. The text, restaged and checked against the prereg.
import hashlib, json
TEXT = f"{ROOT}/wo/text"
step(["-m", "weight_observer.stage_text", "--dest", TEXT], "stage_text")
for n, h in {"train.txt": "aee724fa58bfbdeb3fc6803297fb6bab27b203d7c40b39ddef9b9770e5d52fe5",
             "test.txt": "696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83"}.items():
    assert hashlib.sha256(open(f"{TEXT}/{n}", "rb").read()).hexdigest() == h, n
print("text matches the prereg")

In [ ]:
# 6. Per model: the registered model, windows, plans and results (all hash-checked), then the six arms.
MODELS = [
 [
  "llama3.1-8b",
  "unsloth/Meta-Llama-3.1-8B",
  "e9a141a2091ea561b96483212645a2a05e6f99fc",
  "7dcba79b76b0e185ab48d7e27697db01da6345122b6fd0f000a3d64ae9229db3",
  "2eda697bba5e21a5f25b95c50528b3325025ffafc2c7ef020a1e2211af163223",
  "df98535d971904026b4ad82cb4b50edf939d9bdf0ebdf0405f123d8cd0b81364",
  "fd032948d997c9c568f27dba635255949aecfe49a8d7393b73150bf2a8c71e14"
 ],
 [
  "qwen2.5-3b",
  "Qwen/Qwen2.5-3B",
  "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b",
  "adda89efcc73d3ef9070f4cc151e86fae9b6460e6c26c224a10d2a8fd9798a7b",
  "8c11bf4fd44c94b7c919fb3ebb82bf24a81fb6cc00b5a864df33dbab1f48fd91",
  "23394b0567acb17ebe1911b84cb19dcc54ce6006176dbb1f251768641553983a",
  "fdd222b9f71af8a8f205d0b29fe2bf537f3910307df087978d1f3c788db9b145"
 ],
 [
  "gemma-2-2b",
  "unsloth/gemma-2-2b",
  "25319945f7fd83b8b903e12081777b7eef2ba993",
  "ee9caaca9a27035a72debf59f018165600f23d36438f01615f7ce0ae590633a1",
  "29e2864161ba47b9b266af5839a76e3632544e4079876090a4327b87512a7645",
  "7e723dd907f38c7ec2cca2dd75caec5970842be1288bcc22cd1cd59da2abeb1a",
  "63bdebed0e65b012ecbde9aa281dfefc42c3e22993014369e2f59007f9da485c"
 ]
]
ONLY = os.environ.get("NB_MODELS", "")  # glue test: a subset
for key, hf, rev, calib_sha, eval_sha, plans_sha, reg_sha in MODELS:
    if ONLY and key not in ONLY.split(","):
        continue
    print("=" * 20, key, flush=True)
    M = f"{ROOT}/wo/models/{key}"
    step(["-m", "weight_observer.stage_models", "--model-id", hf, "--revision", rev, "--dest", M], f"stage {key}")
    assert json.load(open(f"{M}/STAGED.json"))["revision"] == rev
    W = f"{ROOT}/wo/windows/{key}"
    if not os.path.exists(f"{W}/windows.pt"):
        step(["-m", "weight_observer.codec_run", "windows", "--model-path", M, "--text", TEXT, "--out", W], f"windows {key}")
    h = json.load(open(f"{W}/hashes.json"))
    assert h["calibration_windows"]["sha256"] == calib_sha, f"{key}: calibration windows differ"
    assert h["evaluation_windows"]["sha256"] == eval_sha, f"{key}: scored windows differ"
    R = f"benchmarks/weight_observer/results/codec/{key}"
    IN = {}
    for f, sha in (("arms.json", plans_sha), ("arms_results.jsonl", reg_sha)):
        blob = subprocess.run(["git", "-C", CODE, "show", f"{C}:{R}/{f}"], capture_output=True, check=True).stdout
        assert hashlib.sha256(blob).hexdigest() == sha, f"{key}: {f} differs from the registered one"
        IN[f] = f"{ROOT}/{key}.registered_{f}"
        open(IN[f], "wb").write(blob)
    print(key, "model, windows, plans and registered results all match", flush=True)
    if not TEST:
        step(["-m", "weight_observer.packed_check", "run", "--model-path", M, "--windows", f"{W}/windows.pt",
              "--arms-file", IN["arms.json"], "--registered", IN["arms_results.jsonl"], "--out", f"{OUT}/{key}"],
             f"packed_check {key}")
print("STEP6_DONE")

In [ ]:
# 7. Check and hand-back.
ARMS = ["gptq_u3", "gptq_u4", "gptq_f3", "gptq_f4", "gptq_frtn3", "gptq_frtn4"]
if not TEST:
    for key, *_ in MODELS:
        rows = [json.loads(l) for l in open(f"{OUT}/{key}/packed_results.jsonl")]
        kl = lambda s: sum(x["kl_sum"] for x in s) / sum(x["tokens"] for x in s)
        for r in rows:
            print(f"{key:12s} {r['arm']:11s} codec {kl(r['codec_seqs']):.5f} decoded {kl(r['decoded_seqs']):.5f} "
                  f"identical to registered: {r['bit_identical_to_registered']}")
        assert sorted(r["arm"] for r in rows) == sorted(ARMS), key
    from google.colab import drive
    drive.flush_and_unmount()
print("DONE: MyDrive/tqp-weights/packed (G: on the laptop)")